# Instance Segmentation Comparison

Linear comparison between Toy Mask2Former (DINOv3 backbone) and Graha/Lunar-FM Mask R-CNN on the same split instance dataset.

In [ ]:
# Notebook imports
# Generated from standard/third-party imports used throughout this notebook.
import sys
from argparse import Namespace
from pathlib import Path


## Config

In [ ]:

NOTEBOOK_DIR = Path.cwd().resolve()
if not (NOTEBOOK_DIR / "instance_seg_comparison.ipynb").exists():
    NOTEBOOK_DIR = (Path.cwd() / "notebooks" / "full_model").resolve()

SIMLINK_DEST = None
DATA_ROOT = Path("/panfs/ccds02/nobackup/projects/lfm/model_inputs/300_300_inputs/full_model_inst_seg_v2")
BASE_OUTPUT_DIR = NOTEBOOK_DIR / "outputs" / "instance_seg_comparison"

DINO_CHECKPOINT = None
TOY_LIGHTNING_CHECKPOINT = None
GRAHA_PRETRAIN_DIR = None
GRAHA_WAC_MODE = "new-wac"  # Use "vis-uv" to reuse pretrained 5-band vis + 2-band uv modalities.
GRAHA_VIS_UV_MERGE_METHOD = "mean"
GRAHA_LIGHTNING_CHECKPOINT = None

TARGET_SIZE = 256
BAND_FILTER = [0, 1, 2, 3, 4, 5, 6]
MAX_TRAIN_SAMPLES = None
MAX_VAL_SAMPLES = None
MAX_TEST_SAMPLES = None
TOY_BATCH_SIZE = 2
TOY_NUM_WORKERS = 10
GRAHA_STATS_BATCH_SIZE = 16
GRAHA_BATCH_SIZE = 2
GRAHA_NUM_WORKERS = 10
MAX_EPOCHS = 1

TOY_LEARNING_RATE = 5.0e-5
TOY_WEIGHT_DECAY = 1.0e-3
TOY_FREEZE_BACKBONE = False
TOY_NORMALIZE_INPUTS = False
TOY_GRADIENT_CLIP_VAL = 1.0
DISABLE_TOY_GRADIENT_CLIPPING = False

GRAHA_BACKBONE_LR = 5.0e-5
GRAHA_HEAD_LR = 2.0e-4
GRAHA_LAYER_DECAY = 0.75
GRAHA_WEIGHT_DECAY = 0.05
GRAHA_WARMUP_STEPS = 500
GRAHA_ANCHOR_SIZES = [[8], [16], [32], [64]]
GRAHA_ANCHOR_ASPECT_RATIOS = [0.5, 1.0, 2.0]
GRAHA_SCORE_THRESHOLD = 0.5

PREDICTION_SPLIT = "val"
PREDICTION_N_SAMPLES = 5
PREDICTION_SCORE_THRESHOLD = 0.5
MASK_SHIFT = (0, 0)

SKIP_TOY_FIT = False
SKIP_GRAHA_FIT = False
NO_FIT = False
SEED = 42

## Environment

In [ ]:

LFM_ROOT = NOTEBOOK_DIR.parents[1]
SCRIPTS_PY_DIR = LFM_ROOT / "scripts" / "python"
SCRIPTS_TASK_DIR = SCRIPTS_PY_DIR / "instance_seg"
for import_path in [SCRIPTS_TASK_DIR, SCRIPTS_PY_DIR, LFM_ROOT]:
    if str(import_path) not in sys.path:
        sys.path.insert(0, str(import_path))

import instance_seg_comparison as workflow
from lfm.full_model.all_tasks.utils.utils import ensure_data_symlink

workflow.graha_workflow.configure_proj_environment()
ensure_data_symlink(SIMLINK_DEST, NOTEBOOK_DIR / "data")

## Build Config

In [ ]:
args = Namespace(
    simlink_dest=SIMLINK_DEST,
    data_root=str(DATA_ROOT) if DATA_ROOT is not None else None,
    base_output_dir=str(BASE_OUTPUT_DIR) if BASE_OUTPUT_DIR is not None else None,
    dino_checkpoint=str(DINO_CHECKPOINT) if DINO_CHECKPOINT is not None else None,
    toy_lightning_checkpoint=str(TOY_LIGHTNING_CHECKPOINT) if TOY_LIGHTNING_CHECKPOINT is not None else None,
    graha_pretrain_dir=str(GRAHA_PRETRAIN_DIR) if GRAHA_PRETRAIN_DIR is not None else None,
    graha_wac_mode=GRAHA_WAC_MODE,
    graha_vis_uv_merge_method=GRAHA_VIS_UV_MERGE_METHOD,
    graha_lightning_checkpoint=str(GRAHA_LIGHTNING_CHECKPOINT) if GRAHA_LIGHTNING_CHECKPOINT is not None else None,
    target_size=TARGET_SIZE,
    band_filter=BAND_FILTER,
    max_train_samples=MAX_TRAIN_SAMPLES,
    max_val_samples=MAX_VAL_SAMPLES,
    max_test_samples=MAX_TEST_SAMPLES,
    toy_batch_size=TOY_BATCH_SIZE,
    toy_num_workers=TOY_NUM_WORKERS,
    graha_stats_batch_size=GRAHA_STATS_BATCH_SIZE,
    graha_batch_size=GRAHA_BATCH_SIZE,
    graha_num_workers=GRAHA_NUM_WORKERS,
    max_epochs=MAX_EPOCHS,
    toy_learning_rate=TOY_LEARNING_RATE,
    toy_weight_decay=TOY_WEIGHT_DECAY,
    toy_freeze_backbone=TOY_FREEZE_BACKBONE,
    toy_normalize_inputs=TOY_NORMALIZE_INPUTS,
    toy_gradient_clip_val=TOY_GRADIENT_CLIP_VAL,
    disable_toy_gradient_clipping=DISABLE_TOY_GRADIENT_CLIPPING,
    graha_backbone_lr=GRAHA_BACKBONE_LR,
    graha_head_lr=GRAHA_HEAD_LR,
    graha_layer_decay=GRAHA_LAYER_DECAY,
    graha_weight_decay=GRAHA_WEIGHT_DECAY,
    graha_warmup_steps=GRAHA_WARMUP_STEPS,
    graha_anchor_sizes=GRAHA_ANCHOR_SIZES,
    graha_anchor_aspect_ratios=GRAHA_ANCHOR_ASPECT_RATIOS,
    graha_score_threshold=GRAHA_SCORE_THRESHOLD,
    prediction_split=PREDICTION_SPLIT,
    prediction_n_samples=PREDICTION_N_SAMPLES,
    prediction_score_threshold=PREDICTION_SCORE_THRESHOLD,
    mask_shift=MASK_SHIFT,
    skip_toy_fit=SKIP_TOY_FIT,
    skip_graha_fit=SKIP_GRAHA_FIT,
    no_fit=NO_FIT,
    seed=SEED,
)

config = workflow.build_config(args)
workflow.validate_paths(config)
config

## Output Directory

In [ ]:
output_dir = workflow.create_timestamped_output_dir(config.base_output_dir)
(output_dir / "checkpoints" / "toy_model").mkdir(parents=True, exist_ok=True)
(output_dir / "checkpoints" / "full_model").mkdir(parents=True, exist_ok=True)
workflow.save_config(config, output_dir)
print("Output directory:", output_dir)

## Train Toy Mask2Former (DINOv3 backbone)

In [ ]:
workflow.toy_workflow.run_toy_workflow(
    config,
    output_dir,
    normalization_modality_info=workflow.get_toy_normalization_modality_info(config),
    epoch_test_suite_callback_cls=workflow.InstanceEpochTestSuiteCallback,
)

## Train Graha/Lunar-FM Mask R-CNN

In [ ]:
workflow.graha_workflow.run_graha_workflow(
    config,
    output_dir,
    validation_plot_callback_cls=workflow.GrahaInstancePlotCallback,
    epoch_test_suite_callback_cls=workflow.InstanceEpochTestSuiteCallback,
)